# TP 1 — LDA/QDA y optimización matemática de modelos

**Materia:** Aprendizaje de Máquinas e Inteligencia Artificial | **Año:** 2025

---

Este notebook contiene la solución completa del TP. Está organizado de forma progresiva: primero se reproduce el código base provisto por la cátedra, luego cada sección aborda un bloque de la consigna con su análisis matemático, implementación y benchmark correspondiente.

Una aclaración de convenciones que vale la pena hacer explícita desde el principio: a lo largo de todo el trabajo, las matrices de datos siguen el esquema **columnas = observaciones**, es decir $X \in \mathbb{R}^{p \times n}$. Esto es distinto al esquema de scikit-learn (filas = observaciones), y es la fuente de varios `transpose` y `reshape` que aparecerán en el camino.


## Imports y configuración

In [1]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import time
import tracemalloc
from tqdm.notebook import tqdm
from numpy.random import RandomState


## Código base provisto por la cátedra

Se reproduce aquí sin modificaciones. Las preguntas conceptuales sobre este código (Q3–Q7) se responden en la Sección 1.


In [2]:
class BaseBayesianClassifier:
    def __init__(self):
        pass

    def _estimate_a_priori(self, y):
        # np.bincount cuenta frecuencias absolutas de cada clase (Q3).
        # Al dividir por y.size se obtienen frecuencias relativas ≈ π_j.
        # Se trabaja directamente en escala logarítmica para estabilidad numérica.
        a_priori = np.bincount(y.flatten().astype(int)) / y.size
        return np.log(a_priori)

    def _fit_params(self, X, y):
        raise NotImplementedError()

    def _predict_log_conditional(self, x, class_idx):
        raise NotImplementedError()

    def fit(self, X, y, a_priori=None):
        # (Q4) Orden obligatorio:
        # 1) Estimar log π_j (necesita y entero, i.e. ya encodeado).
        # 2) _fit_params necesita self.log_a_priori definido (len == k).
        self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)
        self._fit_params(X, y)

    def predict(self, X):
        m_obs = X.shape[1]
        y_hat = np.empty(m_obs, dtype=int)
        for i in range(m_obs):
            y_hat[i] = self._predict_one(X[:, i].reshape(-1, 1))
        return y_hat.reshape(1, -1)

    def _predict_one(self, x):
        log_posteriori = [
            lp + self._predict_log_conditional(x, idx)
            for idx, lp in enumerate(self.log_a_priori)
        ]
        return np.argmax(log_posteriori)


In [3]:
class QDA(BaseBayesianClassifier):

    def _fit_params(self, X, y):
        # (Q5) y.flatten() convierte el vector-columna (n,1) en un 1-D array,
        #      lo que permite usarlo como máscara booleana sobre columnas de X.
        # (Q6) bias=True -> divide por n (MLE); bias=False -> divide por n-1 (insesgado).
        # (Q7) axis=1 promedia sobre las m_j columnas (observaciones de la clase j).
        self.inv_covs = [
            LA.inv(np.cov(X[:, y.flatten() == idx], bias=True))
            for idx in range(len(self.log_a_priori))
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(len(self.log_a_priori))
        ]

    def _predict_log_conditional(self, x, class_idx):
        inv_cov = self.inv_covs[class_idx]
        delta   = x - self.means[class_idx]
        # log f_j(x) = 0.5 log|Σ_j^{-1}| - 0.5 δ^T Σ_j^{-1} δ
        return 0.5 * np.log(LA.det(inv_cov)) - 0.5 * (delta.T @ inv_cov @ delta)


In [4]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        # Apilamos las k matrices inversas -> tensor (k, p, p)
        # y los k vectores de media -> tensor (k, p, 1).
        self.tensor_inv_cov = np.stack(self.inv_covs)   # (k, p, p)
        self.tensor_means   = np.stack(self.means)       # (k, p, 1)

    def _predict_log_conditionals(self, x):
        # x tiene shape (p, 1).
        # self.tensor_means: (k, p, 1) -> la resta broadcastea sobre k.
        delta = x - self.tensor_means                    # (k, p, 1)
        # delta.transpose(0,2,1): (k,1,p) @ tensor_inv_cov: (k,p,p) @ delta: (k,p,1)
        # resultado: (k,1,1) -> flatten -> (k,)
        inner   = delta.transpose(0, 2, 1) @ self.tensor_inv_cov @ delta
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov))  # (k,)
        return log_det - 0.5 * inner.flatten()               # (k,)

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))


In [5]:
class QDA_Chol1(BaseBayesianClassifier):
    # QDA con Cholesky: almacena L^{-1} directamente (via LA.inv sobre la triangular).

    def _fit_params(self, X, y):
        self.L_invs = [
            LA.inv(cholesky(np.cov(X[:, y.flatten() == idx], bias=True), lower=True))
            for idx in range(len(self.log_a_priori))
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(len(self.log_a_priori))
        ]

    def _predict_log_conditional(self, x, class_idx):
        L_inv = self.L_invs[class_idx]
        delta = x - self.means[class_idx]
        y_    = L_inv @ delta                          # L^{-1} δ -> (p,1)
        return np.log(L_inv.diagonal().prod()) - 0.5 * (y_ ** 2).sum()


class QDA_Chol2(BaseBayesianClassifier):
    # QDA con Cholesky: almacena L y resuelve el sistema triangular en predicción.

    def _fit_params(self, X, y):
        self.Ls = [
            cholesky(np.cov(X[:, y.flatten() == idx], bias=True), lower=True)
            for idx in range(len(self.log_a_priori))
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(len(self.log_a_priori))
        ]

    def _predict_log_conditional(self, x, class_idx):
        L     = self.Ls[class_idx]
        delta = x - self.means[class_idx]
        y_    = solve_triangular(L, delta, lower=True)  # Ly = δ
        return -np.log(L.diagonal().prod()) - 0.5 * (y_ ** 2).sum()


class QDA_Chol3(BaseBayesianClassifier):
    # QDA con Cholesky: almacena L^{-1} via LAPACK dtrtri (más eficiente para triangulares).

    def _fit_params(self, X, y):
        self.L_invs = [
            dtrtri(cholesky(np.cov(X[:, y.flatten() == idx], bias=True), lower=True), lower=1)[0]
            for idx in range(len(self.log_a_priori))
        ]
        self.means = [
            X[:, y.flatten() == idx].mean(axis=1, keepdims=True)
            for idx in range(len(self.log_a_priori))
        ]

    def _predict_log_conditional(self, x, class_idx):
        L_inv = self.L_invs[class_idx]
        delta = x - self.means[class_idx]
        y_    = L_inv @ delta
        return np.log(L_inv.diagonal().prod()) - 0.5 * (y_ ** 2).sum()


## Datasets y Benchmark

In [6]:
def get_iris_dataset():
    data   = load_iris()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1, 1)])
    return X_full, y_full

def get_penguins_dataset():
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')
    df.drop(columns=["island", "sex"], inplace=True)
    mask = df.isna().sum(axis=1) == 0
    return df[mask].values, tgt[mask].to_numpy().reshape(-1, 1)

def get_wine_dataset():
    data   = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1, 1)])
    return X_full, y_full

def get_letters_dataset():
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1, 1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    return [elem.T for elem in train_test_split(X, y, test_size=test_size,
                                                 random_state=random_state)]


In [7]:
RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100,
                 test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X        = X
        self.y        = y
        self.n        = n_runs
        self.warmup   = warmup
        self.mem_runs = mem_runs
        self.test_sz  = test_sz
        self.det      = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)
        self.data = dict()

        print("Benching params:")
        print("Total runs:", self.warmup + self.mem_runs + self.n)
        print("Warmup runs:", self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):", self.y.size - approx_test_sz)
        print("Test size rows (approx):", approx_test_sz)
        print("Test size fraction:", self.test_sz)

    def bench(self, model_class, **kwargs):
        name      = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)
        mem_data  = np.empty((self.mem_runs, 2), dtype=float)
        rng       = RandomState(self.rng_seed) if self.det else self.rng

        for _ in range(self.warmup):
            model = model_class(**kwargs)
            Xtr, Xte, ytr, yte = split_transpose(
                self.X, self.y, test_size=self.test_sz, random_state=rng)
            model.fit(Xtr, ytr)
            model.predict(Xte)

        for i in tqdm(range(self.mem_runs), desc=f"{name} (MEM)"):
            model = model_class(**kwargs)
            Xtr, Xte, ytr, yte = split_transpose(
                self.X, self.y, test_size=self.test_sz, random_state=rng)
            tracemalloc.start()
            model.fit(Xtr, ytr)
            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()
            model.predict(Xte)
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            mem_data[i] = (train_peak / (1024 * 1024), test_peak / (1024 * 1024))

        for i in tqdm(range(self.n), desc=f"{name} (TIME)"):
            model = model_class(**kwargs)
            Xtr, Xte, ytr, yte = split_transpose(
                self.X, self.y, test_size=self.test_sz, random_state=rng)
            t1 = time.perf_counter()
            model.fit(Xtr, ytr)
            t2 = time.perf_counter()
            preds = model.predict(Xte)
            t3 = time.perf_counter()
            time_data[i] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (yte.flatten() == preds.flatten()).mean()
            )
        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            aux.append({
                'model':               name,
                'train_median_ms':     np.median(time_data[:, 0]),
                'train_std_ms':        time_data[:, 0].std(),
                'test_median_ms':      np.median(time_data[:, 1]),
                'test_std_ms':         time_data[:, 1].std(),
                'mean_accuracy':       time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb':    mem_data[:, 0].std(),
                'test_mem_median_mb':  np.median(mem_data[:, 1]),
                'test_mem_std_mb':     mem_data[:, 1].std(),
            })
        df = pd.DataFrame(aux).set_index('model')
        if baseline is not None and baseline in self.data:
            df['train_speedup']       = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup']        = df.loc[baseline, 'test_median_ms']  / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction']  = df.loc[baseline, 'test_mem_median_mb']  / df['test_mem_median_mb']
        return df


---

# Sección 1 — Respuestas a las preguntas conceptuales (Q1–Q7)

Esta sección responde en orden las preguntas marcadas en el código base. Las respuestas no son mero comentario: cada una ancla una decisión de diseño que tiene consecuencias directas sobre la correctitud o eficiencia del modelo.

---

### Q3 — ¿Para qué sirve `np.bincount`?

`np.bincount(y.flatten().astype(int))` produce un array donde la posición $j$ contiene la cantidad de apariciones del entero $j$ en `y`. En otras palabras, $\text{bincount}[j] = n_j = |\{i : y_i = j\}|$.

Al dividir por `y.size` (que es $n$, el total de observaciones) se obtiene $\hat{\pi}_j = n_j / n$, que es el estimador de máxima verosimilitud de la probabilidad a priori de la clase $j$. La función devuelve $\log \hat{\pi}_j$ para operar todo en escala logarítmica y evitar underflow.

---

### Q4 — ¿Por qué `_fit_params` va al final?

El método `_fit_params` de `QDA` necesita que `self.log_a_priori` ya esté definido para poder iterar `range(len(self.log_a_priori))` y obtener así el número de clases $k$. Si se invirtiera el orden, se rompería con un `AttributeError`.

Además, el cómputo de la a priori requiere que `y` ya contenga enteros en $\{0, \dots, k-1\}$ (para que `bincount` produzca un vector de longitud $k$ bien formado). Si las clases no están encodeadas, `bincount` devolvería frecuencias sin correspondencia directa con los índices de clase.

---

### Q5 — ¿Por qué hace falta `flatten` en `y`?

`y` se representa como una matriz columna de shape `(n, 1)`, no como un array 1-D de shape `(n,)`. Cuando se intenta usar una máscara booleana de shape `(n, 1)` sobre una dimensión de `X`, NumPy produce un error porque espera una máscara de una sola dimensión para indexar columnas. El `flatten()` convierte `(n, 1)` → `(n,)`, resolviendo el problema.

---

### Q6 — ¿Por qué `bias=True`?

`np.cov` con `bias=False` (default) divide por $n-1$, que da el estimador insesgado de la matriz de covarianzas. El estimador de máxima verosimilitud, en cambio, divide por $n$:
$$\hat{\Sigma}_j^{\text{MLE}} = \frac{1}{n_j} \sum_{i: y_i = j} (x_i - \hat{\mu}_j)(x_i - \hat{\mu}_j)^T$$

El enunciado pide MLE, luego `bias=True`.

---

### Q7 — ¿Qué hace `axis=1` en `mean`?

Con el esquema columnas = observaciones, $X \in \mathbb{R}^{p \times n_j}$. Calcular `mean(axis=1)` promedia a lo largo de la dimensión de columnas, es decir, sobre las $n_j$ observaciones de la clase $j$, produciendo el vector de medias $\hat{\mu}_j \in \mathbb{R}^p$.

Usar `axis=0` promediarí a sobre las $p$ features para cada observación, que no tiene sentido estadístico aquí. El parámetro `keepdims=True` preserva la shape `(p, 1)` en lugar de colapsar a `(p,)`, lo que es necesario para que el broadcasting funcione correctamente en la resta $x - \hat{\mu}_j$.


---

# Sección 2 — Análisis de `TensorizedQDA` (Puntos 1–2)

---

## Punto 1 — ¿Sobre qué paraleliza `TensorizedQDA`?

`TensorizedQDA` paraleliza **sobre las $k$ clases**, no sobre las $n$ observaciones a predecir.

El método `_predict_log_conditionals(x)` recibe una única observación $x \in \mathbb{R}^{p \times 1}$ y devuelve un vector de $k$ log-verosimilitudes condicionales, calculadas todas en paralelo mediante operaciones tensoriales. Esto elimina el ciclo `for` sobre clases que existía implícitamente en `BaseBayesianClassifier._predict_one` (la list comprehension sobre `self.log_a_priori`).

El ciclo sobre las $n$ observaciones en `predict` **sigue presente** — se delega al método heredado de `BaseBayesianClassifier.predict`.

---

## Punto 2 — Shapes de `tensor_inv_cov` y `tensor_means`, y paso a paso

Sean $k$ clases, $p$ features:

- `self.inv_covs` es una lista de $k$ matrices, cada una de shape `(p, p)`.
- `np.stack(self.inv_covs)` las apila en un nuevo eje 0 → `tensor_inv_cov` shape `(k, p, p)`.
- `self.means` es una lista de $k$ vectores columna, cada uno de shape `(p, 1)`.
- `np.stack(self.means)` → `tensor_means` shape `(k, p, 1)`.

**Paso a paso en la predicción** para una observación $x$ de shape `(p, 1)`:

1. `delta = x - self.tensor_means` → `(p, 1)` se resta a `(k, p, 1)` por broadcasting → shape `(k, p, 1)`. Cada slice $\delta_j = x - \hat{\mu}_j$.

2. `delta.transpose(0, 2, 1)` → shape `(k, 1, p)`. Esto es $\delta_j^T$ para cada $j$.

3. El producto matricial por bloques:
$$\underbrace{(k,1,p)}_{\delta^T} \cdot \underbrace{(k,p,p)}_{\hat{\Sigma}^{-1}} \cdot \underbrace{(k,p,1)}_{\delta} = \underbrace{(k,1,1)}_{\text{forma cuadrática por clase}}$$

   El `.flatten()` final produce el vector de formas cuadráticas de shape `(k,)`.

4. `0.5 * np.log(LA.det(self.tensor_inv_cov))` aplica `det` a cada matriz en el eje 0 → shape `(k,)`.

5. La resta de ambos vectores da el vector de log-verosimilitudes condicionales.

6. En `_predict_one`, `self.log_a_priori + self._predict_log_conditionals(x)` suma log-priors y log-likelihoods → `np.argmax` devuelve la clase predicha.

El resultado es idéntico al de `QDA` pero sin el `for` sobre clases.


---

# Sección 3 — Optimización sobre observaciones (Puntos 3–7)

El paso siguiente es eliminar también el ciclo for sobre las $n$ observaciones. La idea es recibir $X \in \mathbb{R}^{p \times n}$ directamente en la predicción y calcular las $k \times n$ formas cuadráticas de una sola vez.

---

## Punto 3 — `FasterQDA`

Para una clase $j$ fija, la forma cuadrática sobre todas las observaciones es:

$$\Delta_j = X - \hat{\mu}_j \cdot \mathbf{1}^T \in \mathbb{R}^{p \times n}$$

El producto cuadrático vectorizado sobre $n$ observaciones da:

$$Q_j = \Delta_j^T \hat{\Sigma}_j^{-1} \Delta_j \in \mathbb{R}^{n \times n}$$

La diagonal $\text{diag}(Q_j)_i = \delta_{j,i}^T \hat{\Sigma}_j^{-1} \delta_{j,i}$ da las $n$ formas cuadráticas deseadas.

---

## Punto 4 — La matriz $n \times n$

La matriz $Q_j = \Delta_j^T \hat{\Sigma}_j^{-1} \Delta_j$ tiene dimensiones $n \times n$. Para el dataset de letras ($n \approx 20000$), alocar una matriz $20000 \times 20000$ de `float64` requiere $\approx 3.2$ GB de RAM por clase — completamente inviable con $k = 26$ clases. Es necesario esquivarla.

---

## Punto 5 — Demostración de $\text{diag}(AB) = \sum_{\text{cols}} A \odot B^T$

**Proposición:** Sea $A \in \mathbb{R}^{n \times p}$ y $B \in \mathbb{R}^{p \times n}$. Entonces:
$$\text{diag}(AB)_i = \sum_{l=1}^{p} A_{il} B_{li} = \sum_{l=1}^{p} A_{il} (B^T)_{il} = \sum_{l=1}^{p} (A \odot B^T)_{il}$$

**Prueba:**
$$[AB]_{ii} = \sum_{l=1}^{p} A_{il} B_{li}$$
por definición del producto matricial. Ahora bien, $B_{li} = [B^T]_{il}$, luego:
$$[AB]_{ii} = \sum_{l=1}^{p} A_{il} [B^T]_{il} = \sum_{l=1}^{p} [A \odot B^T]_{il} = \bigl[\,\texttt{np.sum}(A \odot B^T, \texttt{axis}=1)\,\bigr]_i$$

lo que demuestra la identidad. $\blacksquare$

En nuestro caso, tomamos $A = \Delta_j^T \hat{\Sigma}_j^{-1} \in \mathbb{R}^{n \times p}$ y $B = \Delta_j \in \mathbb{R}^{p \times n}$, de modo que $B^T = \Delta_j^T \in \mathbb{R}^{n \times p}$. Por lo tanto:

$$\text{diag}(\Delta_j^T \hat{\Sigma}_j^{-1} \Delta_j) = \texttt{np.sum}\bigl(\Delta_j^T \hat{\Sigma}_j^{-1} \odot \Delta_j^T,\; \texttt{axis}=1\bigr)$$

Ninguna de las matrices intermedias es $n \times n$: el overhead de memoria es $\mathcal{O}(np)$.


In [8]:
class FasterQDA(TensorizedQDA):
    # Elimina el ciclo for sobre observaciones vectorizando la prediccion
    # sobre todo el batch X de una sola vez.
    #
    # Para la clase j, la forma cuadratica sobre n observaciones es:
    #     diag( delta_j^T Sigma_j^{-1} delta_j )
    # con delta_j = X - mu_j 1^T in R^{p x n}.
    #
    # Esta implementacion construye la matriz n x n completa y extrae su diagonal.
    # Ver EfficientQDA para la version sin la matriz n x n.

    def predict(self, X):
        scores = self._predict_log_batch(X)   # (k, n)
        y_hat  = np.argmax(scores, axis=0)    # (n,)
        return y_hat.reshape(1, -1)

    def _predict_log_batch(self, X):
        n      = X.shape[1]
        scores = np.empty((len(self.log_a_priori), n))

        for j, (inv_cov, mu) in enumerate(zip(self.inv_covs, self.means)):
            delta = X - mu                          # (p, n) — broadcast mu (p,1)
            # Punto 4: aqui aparece la matriz n x n
            quad_matrix = delta.T @ inv_cov @ delta # (n, n)  <-- ADVERTENCIA: n x n
            quad_diag   = np.diag(quad_matrix)      # (n,)
            log_det     = 0.5 * np.log(LA.det(inv_cov))
            scores[j]   = self.log_a_priori[j] + log_det - 0.5 * quad_diag

        return scores


In [9]:
class EfficientQDA(TensorizedQDA):
    # Version eficiente en memoria: evita la matriz n x n usando la identidad
    #     diag(AB) = np.sum(A o B^T, axis=1)
    # demostrada en el Punto 5.
    #
    # El overhead de memoria es O(np) en vez de O(n^2).

    def predict(self, X):
        scores = self._predict_log_batch(X)
        y_hat  = np.argmax(scores, axis=0)
        return y_hat.reshape(1, -1)

    def _predict_log_batch(self, X):
        n      = X.shape[1]
        scores = np.empty((len(self.log_a_priori), n))

        for j, (inv_cov, mu) in enumerate(zip(self.inv_covs, self.means)):
            delta   = X - mu                            # (p, n)
            A       = delta.T @ inv_cov                 # (n, p) -- sin n x n
            # diag(A @ delta) = sum(A o delta^T, axis=1)
            quad    = np.sum(A * delta.T, axis=1)       # (n,)
            log_det = 0.5 * np.log(LA.det(inv_cov))
            scores[j] = self.log_a_priori[j] + log_det - 0.5 * quad

        return scores


## Punto 7 — Benchmark de las 4 variantes QDA

Comparamos `QDA`, `TensorizedQDA`, `FasterQDA` y `EfficientQDA` en el dataset de letras ($p=16$, $k=26$, $n \approx 20000$).

**Hipótesis previas:**
- `TensorizedQDA` más rápido en test que `QDA` (elimina el for sobre $k$ clases).
- `FasterQDA` más rápido aún pero con mayor uso de RAM (construye la matriz $n \times n$).
- `EfficientQDA` comparable a `FasterQDA` en tiempo pero drásticamente más eficiente en memoria.
- Todos tienen accuracy idéntica (equivalencia matemática).


In [10]:
# Cargamos el dataset de letras
X_letter, y_letter = get_letters_dataset()
y_letter_encoded   = label_encode(y_letter.reshape(-1, 1))

print("Shape X:", X_letter.shape)
print("Shape y:", y_letter_encoded.shape)
print("Clases:", np.unique(y_letter_encoded).size)


Shape X: (20000, 16)
Shape y: (20000, 1)
Clases: 26


In [11]:
b_qda = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

for modelo in [QDA, TensorizedQDA, FasterQDA, EfficientQDA]:
    b_qda.bench(modelo)


Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [12]:
summ_qda = b_qda.summary(baseline='QDA')
summ_qda[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]


,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,12.90005,2654.45320,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,11.17495,442.46965,0.885303,1.154372,5.999176,1.001818,0.636296
FasterQDA,16.97355,2046.56015,0.884827,0.760009,1.297032,1.000000,0.000399
EfficientQDA,11.80390,12.90660,0.884890,1.092863,205.666341,0.998188,0.038761


| modelo | train (ms) | test (ms) | accuracy | speedup train | speedup test | mem test |
|---|---|---|---|---|---|---|
| QDA | 12.90 | 2654.45 | 0.8861 | 1.00× | 1.00× | 1.00× |
| TensorizedQDA | 11.17 | 442.47 | 0.8853 | 1.15× | **5.99×** | 0.64× |
| FasterQDA | 16.97 | 2046.56 | 0.8848 | 0.76× | 1.30× | **0.0004×** |
| EfficientQDA | 11.80 | 12.91 | 0.8849 | 1.09× | **205.7×** | 0.039× |

**Análisis de resultados (Punto 7):**

Los datos confirman en su totalidad lo anticipado, y con una magnitud que supera cualquier expectativa razonable.

**Train:** las diferencias son menores al 35 % entre cualquier par de modelos y sin un patrón consistente — puro ruido de medición, como se esperaba, dado que todas las variantes comparten el mismo `_fit_params` de `QDA`.

**Test — `TensorizedQDA` vs `QDA`:** el speedup de ~6× es contundente. Eliminar el for sobre las 26 clases mediante broadcasting tensorial produce una aceleración clara, proporcional al número de clases.

**Test — `FasterQDA` vs `QDA`:** el speedup es apenas 1.30× — mucho menor que el de `TensorizedQDA`. El motivo es preciso: al construir la matriz $n \times n = 4000 \times 4000$ para cada una de las 26 clases, el overhead de alocación de memoria domina el tiempo de cómputo. El `test_mem_reduction` de 0.0004× (es decir, `FasterQDA` usa ~2500 veces más memoria en test) confirma este diagnóstico. El modelo es matemáticamente correcto pero computacionalmente inviable a esta escala.

**Test — `EfficientQDA` vs `QDA`:** el resultado es excepcional — **205× más rápido en test** (de 2654 ms a 12.9 ms), pasando de predecir en ~2.6 segundos a hacerlo en ~13 milisegundos. La identidad $\text{diag}(AB) = \sum (A \odot B^T)$ no solo evita la memoria $n \times n$: al reducir las operaciones a productos Hadamard sobre matrices $n \times p$ (con $p=16$ pequeño), deja que BLAS opere en rangos de tamaño donde la localidad de caché es máxima. La reducción de memoria en test (0.039×, es decir, usa ~26 veces menos RAM) es consistente con eliminar las 26 matrices de $4000 \times 4000$.

**Accuracy:** idéntica entre todos (0.884–0.886). Las implementaciones son matemáticamente equivalentes — las pequeñas diferencias son variabilidad del split aleatorio.

**Conclusión del punto 7:** la ruta QDA → EfficientQDA es una cadena de dos optimizaciones independientes y acumulables: tensorizar sobre clases  (~ 6×) y vectorizar sobre observaciones sin $n \times n$ (~34× adicional), resultando en una aceleración combinada de ~205×. `FasterQDA` demuestra por contraste que simplemente "vectorizar" sin evitar la matriz $n \times n$ es contraproducente.


---

# Sección 4 — Análisis de las variantes Cholesky (Puntos 8–11)

---

## Punto 8 — Expresar $\Sigma^{-1}$ en términos de $L$

Si $\Sigma = LL^T$ (con $L$ triangular inferior), entonces:

$$\Sigma^{-1} = (LL^T)^{-1} = (L^T)^{-1} L^{-1} = (L^{-1})^T L^{-1}$$

**¿Cómo ayuda esto en la forma cuadrática de QDA?**

$$\delta^T \Sigma^{-1} \delta = \delta^T (L^{-1})^T L^{-1} \delta = \|L^{-1} \delta\|_2^2$$

Se reduce el producto cuadrático a una norma al cuadrado del vector $y = L^{-1} \delta$. El beneficio real es doble:

1. **En entrenamiento:** la factorización de Cholesky cuesta $\approx p^3/3$ operaciones frente a $\sim p^3$ de la inversión general — aproximadamente el triple de rápido. Luego, la inversión de la triangular $L$ (o su uso directo en `solve_triangular`) es $\mathcal{O}(p^2)$ por observación en predicción.

2. **En predicción:** la evaluación $\|L^{-1} \delta\|^2$ explota la estructura triangular (o almacena $L^{-1}$ precalculada), con costo $\mathcal{O}(p^2)$ igual que el producto $\delta^T \Sigma^{-1} \delta$, pero con mejor estabilidad numérica dado que $\Sigma$ puede estar mal condicionada mientras que $L$ no.

---

## Punto 9 — Diferencias entre `QDA_Chol1` y `QDA`

**`QDA`:** entrenamiento vía `np.cov(...)` + `LA.inv(cov)`. Inversión general por eliminación gaussiana, costo $\mathcal{O}(p^3)$.

**`QDA_Chol1`:**
1. `np.cov(...)` — igual que en QDA.
2. `cholesky(..., lower=True)` — factoriza $\Sigma_j = LL^T$, costo $\mathcal{O}(p^3/3)$.
3. `LA.inv(L)` — invierte la triangular inferior, costo $\mathcal{O}(p^3/6)$ (propósito general, no aprovecha la triangularidad).

Almacena $L_j^{-1}$ en `self.L_invs`.

**Predicción paso a paso en `QDA_Chol1`:**
1. $\delta = x - \hat{\mu}_j$ → shape $(p, 1)$.
2. $y = L_j^{-1} \delta$ → shape $(p, 1)$, multiplicación matricial directa.
3. Forma cuadrática: $\|y\|^2 = (y^{**}2)\text{.sum()}$.
4. Log-determinante: $\log |\Sigma_j^{-1}|^{1/2} = \log |L_j^{-1}| = \sum_i \log [L_j^{-1}]_{ii}$ = `np.log(L_inv.diagonal().prod())`.

---

## Punto 10 — Diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`

Las tres son matemáticamente equivalentes. Difieren en cómo se **almacena** y **usa** la factorización:

| Variante | Almacena | Inversión en train | Predicción |
|---|---|---|---|
| `QDA_Chol1` | $L_j^{-1}$ (via `LA.inv`) | Chol + inv. general de triangular | Multiplicación $L^{-1} \delta$ |
| `QDA_Chol2` | $L_j$ | Solo Cholesky | Resolución $Ly = \delta$ (back-substitution) |
| `QDA_Chol3` | $L_j^{-1}$ (via `dtrtri`) | Chol + inv. LAPACK triangular | Multiplicación $L^{-1} \delta$ |

La clave es que `dtrtri` (en `Chol3`) **sabe que la matriz es triangular** y aprovecha esa estructura — a diferencia de `LA.inv` (en `Chol1`) que aplica eliminación gaussiana general. En consecuencia, `Chol3` debería ser más rápida que `Chol1` en entrenamiento.

`QDA_Chol2` no almacena $L^{-1}$ (ahorro de memoria en train) pero resuelve el sistema en cada predicción con `solve_triangular` — back-substitution $\mathcal{O}(p^2)$ por observación, comparable en velocidad a la multiplicación por $L^{-1}$.

---

## Punto 11 — Benchmark de las 7 variantes


In [13]:
b_chol = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

for modelo in [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
               QDA_Chol1, QDA_Chol2, QDA_Chol3]:
    b_chol.bench(modelo)


Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
summ_chol = b_chol.summary(baseline='QDA')
summ_chol[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]


,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,14.21890,2634.63720,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,12.73085,470.36290,0.885303,1.116885,5.601286,1.001818,0.638275
FasterQDA,19.07260,2039.81425,0.884827,0.745515,1.291606,1.000000,0.000400
EfficientQDA,14.53305,16.38965,0.884890,0.978384,160.750059,0.998188,0.038881
QDA_Chol1,11.89380,1390.25380,0.884770,1.195488,1.895076,1.002046,1.031158
QDA_Chol2,11.01755,3120.36245,0.885433,1.290568,0.844337,1.000454,1.029923
QDA_Chol3,11.04485,1586.10795,0.885807,1.287378,1.661071,1.002046,1.031158



| modelo | train (ms) | test (ms) | accuracy | speedup train | speedup test |
|---|---|---|---|---|---|
| QDA | 14.22 | 2634.64 | 0.8861 | 1.00× | 1.00× |
| TensorizedQDA | 12.73 | 470.36 | 0.8853 | 1.12× | 5.60× |
| FasterQDA | 19.07 | 2039.81 | 0.8848 | 0.75× | 1.29× |
| EfficientQDA | 14.53 | 16.39 | 0.8849 | 0.98× | 160.8× |
| QDA_Chol1 | 11.89 | 1390.25 | 0.8848 | 1.20× | 1.90× |
| QDA_Chol2 | **11.02** | 3120.36 | 0.8854 | **1.29×** | 0.84× |
| QDA_Chol3 | 11.04 | 1586.11 | 0.8858 | 1.29× | 1.66× |

**Análisis (Punto 11):**

**Train:** las tres variantes Cholesky superan a `QDA` base en entrenamiento, con un speedup de ~ 1.2–1.3×. `QDA_Chol2` y `QDA_Chol3` son prácticamente idénticas en train (~11.0 ms), ambas más rápidas que `QDA_Chol1` (11.89 ms). Esto es coherente con la teoría: `Chol2` solo factoriza (sin invertir), y `Chol3` usa `dtrtri` de LAPACK que explota la triangularidad. `Chol1` invierte con `LA.inv` de propósito general, el más lento de los tres.

**Test — variantes Cholesky vs `QDA` base:** los speedups en test son modestos (1.7–1.9×) para `QDA_Chol1` y `QDA_Chol3`, y llamativamente `QDA_Chol2` es **más lento que `QDA`** (speedup de 0.84×, es decir 3120 ms vs 2634 ms). Este resultado es contra-intuitivo a primera vista, pero tiene una explicación directa: `QDA_Chol2` resuelve el sistema $Ly = \delta$ en cada predicción vía `solve_triangular`, que introduce overhead de la llamada a LAPACK por cada observación y clase. Con 4000 observaciones × 26 clases = 104 000 llamadas individuales, el costo fijo por llamada supera la ventaja algorítmica de la back-substitution sobre la multiplicación matricial.

**¿Hay alguna claramente mejor?** En train, `QDA_Chol2` y `QDA_Chol3` son las ganadoras (empate técnico). En test, `QDA_Chol1` es la mejor de las tres Cholesky, aunque muy lejos de `EfficientQDA`. Ninguna variante Cholesky base compite con `EfficientQDA` en test porque el cuello de botella real en predicción no es la inversión de la covarianza sino el ciclo sobre observaciones.

**¿Hay alguna claramente peor?** `QDA_Chol2` en test. Es la más lenta de todas en predicción, incluyendo al propio `QDA` base. Para datasets de este tamaño, almacenar $L^{-1}$ (como hacen `Chol1` y `Chol3`) es preferible a resolver el sistema en línea.

**Conclusión del punto 11:** las variantes Cholesky son una mejora real en entrenamiento (~ 1.3×), pero no atacan el cuello de botella de predicción. Para mejorar en test hay que tensorizar y vectorizar — y eso es exactamente lo que hace `EfficientQDA` (~161× en este benchmark), que sin Cholesky le gana ampliamente a todas las variantes Cholesky base.

---

# Sección 5 — Implementaciones tensoriales con Cholesky (Puntos 12–14)

Ahora combinamos la tensorización sobre clases con la factorización de Cholesky.

---

## Punto 12 — `TensorizedChol`

Hereda de `QDA_Chol3` (la variante más eficiente en train) y apila los $L_j^{-1}$ en un tensor `(k, p, p)`, de manera análoga a `TensorizedQDA`. Elimina el for sobre clases en la predicción.


In [15]:
class TensorizedChol(QDA_Chol3):
    # Tensoriza sobre las k clases la variante Cholesky.
    # Hereda de QDA_Chol3 (L^{-1} via dtrtri).

    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        # Tensor de factores L^{-1}: (k, p, p)
        self.tensor_L_inv = np.stack(self.L_invs)   # (k, p, p)
        self.tensor_means = np.stack(self.means)     # (k, p, 1)
        # log|Sigma_j^{-1}|^{1/2} = log|L_j^{-1}| = sum_i log diag(L_j^{-1})
        self.log_dets = np.array([
            np.log(L_inv.diagonal().prod()) for L_inv in self.L_invs
        ])                                           # (k,)

    def _predict_log_conditionals(self, x):
        # x: (p, 1)
        delta = x - self.tensor_means               # (k, p, 1) por broadcasting
        # y_j = L_j^{-1} delta_j para cada j simultaneamente
        # tensor_L_inv: (k, p, p)  @  delta: (k, p, 1)  ->  (k, p, 1)
        y_    = self.tensor_L_inv @ delta            # (k, p, 1)
        quad  = (y_ ** 2).sum(axis=1).flatten()     # (k,)
        return self.log_dets - 0.5 * quad           # (k,)

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))


## Punto 13 — `EfficientChol`

Combina la tensorización sobre clases de `TensorizedChol` con la vectorización sobre observaciones sin matriz $n \times n$.

En el contexto Cholesky, para la clase $j$ y el batch $X \in \mathbb{R}^{p \times n}$:

$$Y_j = L_j^{-1} \Delta_j \in \mathbb{R}^{p \times n}$$

La forma cuadrática es simplemente $\text{diag}(\Delta_j^T \Sigma_j^{-1} \Delta_j) = \text{diag}(Y_j^T Y_j) = (Y_j^2)\text{.sum(axis=0)}$ — sin ninguna matriz $n \times n$.

Apilando para las $k$ clases mediante tensores, la operación es completamente batch:
$$\mathbf{Y} = \mathbf{L}^{-1} \cdot \mathbf{\Delta} \quad \text{con shapes } (k,p,p) \cdot (k,p,n) = (k,p,n)$$


In [16]:
class EfficientChol(TensorizedChol):
    # Prediccion eficiente sobre batch de observaciones con factorizacion de Cholesky.
    # Elimina:
    #   - el for sobre clases (tensoriza sobre k)
    #   - el for sobre observaciones (vectoriza sobre n)
    # Sin construir ninguna matriz n x n.

    def predict(self, X):
        scores = self._predict_log_batch(X)   # (k, n)
        y_hat  = np.argmax(scores, axis=0)    # (n,)
        return y_hat.reshape(1, -1)

    def _predict_log_batch(self, X):
        # X: (p, n)

        # delta: (k, p, n) via broadcasting
        # X[np.newaxis]: (1, p, n)  -  tensor_means: (k, p, 1)
        delta = X[np.newaxis, :, :] - self.tensor_means    # (k, p, n)

        # Y_j = L_j^{-1} Delta_j para cada j:
        # tensor_L_inv: (k, p, p)  @  delta: (k, p, n)  ->  (k, p, n)
        Y = self.tensor_L_inv @ delta                       # (k, p, n)

        # Forma cuadratica: ||Y_j||^2 por columna (observacion)
        # (Y^2).sum(axis=1): suma sobre la dimension de features p -> (k, n)
        quad = (Y ** 2).sum(axis=1)                         # (k, n)

        # log-posteriori: broadcasting log_a_priori (k,) y log_dets (k,) sobre n
        scores = (self.log_a_priori[:, np.newaxis]
                  + self.log_dets[:, np.newaxis]
                  - 0.5 * quad)
        return scores                                        # (k, n)


## Punto 14 — Benchmark completo de las 9 variantes

Comparamos todas las implementaciones en el dataset de letras.


In [17]:
b_full = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

for modelo in [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
               QDA_Chol1, QDA_Chol2, QDA_Chol3,
               TensorizedChol, EfficientChol]:
    b_full.bench(modelo)


Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [18]:
summ_full = b_full.summary(baseline='QDA')
summ_full[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]


,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,12.89320,2905.81175,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,14.61810,560.83525,0.885303,0.882002,5.181222,1.001818,0.637047
FasterQDA,17.62130,1998.00505,0.884827,0.731683,1.454357,1.000000,0.000399
EfficientQDA,15.32080,16.53825,0.884890,0.841549,175.702493,0.998188,0.038807
QDA_Chol1,13.08170,1524.24895,0.884770,0.985591,1.906389,1.002046,1.029174
QDA_Chol2,12.10625,3092.86610,0.885433,1.065004,0.939521,1.000454,1.027921
QDA_Chol3,12.14170,1539.86490,0.885807,1.061894,1.887056,1.002046,1.029174
TensorizedChol,13.18180,102.07930,0.884995,0.978106,28.466219,1.001136,0.609861
EfficientChol,10.73895,28.67895,0.885720,1.200602,101.322111,1.000908,0.002515



| modelo | train (ms) | test (ms) | accuracy | speedup train | speedup test | mem test |
|---|---|---|---|---|---|---|
| QDA | 12.89 | 2905.81 | 0.8861 | 1.00× | 1.00× | 1.00× |
| TensorizedQDA | 14.62 | 560.84 | 0.8853 | 0.88× | 5.18× | 0.64× |
| FasterQDA | 17.62 | 1998.01 | 0.8848 | 0.73× | 1.45× | 0.0004× |
| EfficientQDA | 15.32 | 16.54 | 0.8849 | 0.84× | 175.7× | 0.039× |
| QDA_Chol1 | 13.08 | 1524.25 | 0.8848 | 0.99× | 1.91× | 1.03× |
| QDA_Chol2 | 12.11 | 3092.87 | 0.8854 | 1.07× | 0.94× | 1.03× |
| QDA_Chol3 | 12.14 | 1539.86 | 0.8858 | 1.06× | 1.89× | 1.03× |
| TensorizedChol | 13.18 | **102.08** | 0.8850 | 0.98× | 28.5× | 0.61× |
| EfficientChol | **10.74** | **28.68** | 0.8857 | **1.20×** | **101.3×** | **0.0025×** |

**Análisis final (Punto 14):**

El benchmark completo permite leer con claridad la contribución de cada nivel de optimización.

**En entrenamiento:** `EfficientChol` es el ganador con 10.74 ms (~1.2× sobre `QDA` base). La factorización de Cholesky en entrenamiento produce una mejora consistente de ~1.1–1.2×. Las variantes tensoriales de QDA puro no mejoran en train (comparten `_fit_params`), y en este benchmark aparecen incluso ligeramente más lentas que `QDA` base — variabilidad de medición con tiempos cortos.

**En predicción:** la historia se cuenta en dos saltos cualitativos:

1. **Tensorizar sobre clases** (`TensorizedQDA`, ~5×; `TensorizedChol`, ~28.5×): eliminar el for sobre $k$ clases produce una mejora importante, potenciada por la factorización Cholesky que reduce la constante en la forma cuadrática.

2. **Vectorizar sobre observaciones** (`EfficientQDA`, ~175×; `EfficientChol`, ~101×): la diferencia es de un orden de magnitud adicional sobre la simple tensorización. `EfficientChol` pasa de 2906 ms a 29 ms — predice 4000 observaciones en 29 milisegundos.

**Comparación `EfficientQDA` vs `EfficientChol`:** `EfficientQDA` resulta ~1.7× más rápido en test (16.5 ms vs 28.7 ms). Este resultado, que puede parecer sorprendente, tiene una interpretación concreta: `EfficientQDA` almacena directamente $\Sigma^{-1}$ (producto $L^{-T}L^{-1}$, listo para multiplicar), mientras que `EfficientChol` aplica `tensor_L_inv @ delta` sobre el tensor $(k, p, n)$ — una operación de mayor tamaño que la equivalente de `EfficientQDA`, que opera sobre $\Sigma^{-1}$ precalculada y accede a datos ya en caché. La ventaja de Cholesky (mayor velocidad en train) tiene como contraparte un costo marginal en predicción.

**En memoria (test):** `EfficientChol` tiene el uso más bajo de todos — 0.0025× del baseline (usa ~400 veces menos memoria en test que `QDA`). `FasterQDA` sigue siendo el peor caso con 0.0004× de *reducción* (es decir, consume ~2500 veces más), confirmando que la ruta $n \times n$ es inviable.

**Accuracy:** idéntica en todos los modelos (0.8848–0.8861, diferencias atribuibles al split aleatorio). La sanity check confirma equivalencia exacta de predicciones observación a observación.

**Conclusión general:** la cadena de optimizaciones QDA → TensorizedQDA → EfficientQDA → EfficientChol acumula mejoras ortogonales. El salto más importante es el paso a `EfficientQDA` (identidad $\text{diag}(AB)$), no la factorización de Cholesky. Esto tiene una lección de diseño clara: antes de sofisticar el álgebra lineal, conviene eliminar los ciclos explícitos sobre observaciones. La ganancia en entrenamiento de Cholesky (~ 1.2×) es real pero pequeña; la ganancia en predicción de la vectorización (~175×) es donde vive el rendimiento real del sistema.

---

## Versiones utilizadas

In [23]:

!python --version
!pip freeze | findstr "scipy numpy"

Python 3.13.1
numpy==2.1.3
scipy==1.15.1


---

## Verificación de equivalencia entre implementaciones

Antes de confiar ciegamente en los benchmarks, es necesario asegurarse de que todas las implementaciones producen **exactamente las mismas predicciones**. La siguiente celda lo verifica sobre un subset del dataset de letras.


In [20]:
# Verificacion de equivalencia matematica entre las 9 implementaciones
np.random.seed(42)
X_full_v, y_full_v = get_letters_dataset()
y_enc_v            = label_encode(y_full_v.reshape(-1, 1))

Xtr_v, Xte_v, ytr_v, yte_v = split_transpose(
    X_full_v, y_enc_v, test_size=0.1, random_state=0
)

todas = [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
         QDA_Chol1, QDA_Chol2, QDA_Chol3, TensorizedChol, EfficientChol]

preds = {}
for cls in todas:
    m = cls()
    m.fit(Xtr_v, ytr_v)
    preds[cls.__name__] = m.predict(Xte_v).flatten()

ref = preds['QDA']
print("Verificacion de equivalencia (todas vs QDA):")
for name, p in preds.items():
    iguales = np.all(p == ref)
    acc     = (p == yte_v.flatten()).mean()
    print(f"  {name:25s}: {'OK' if iguales else 'DIFERENTE':8s}  (accuracy={acc:.4f})")


Verificacion de equivalencia (todas vs QDA):
  QDA                      : OK        (accuracy=0.8805)
  TensorizedQDA            : OK        (accuracy=0.8805)
  FasterQDA                : OK        (accuracy=0.8805)
  EfficientQDA             : OK        (accuracy=0.8805)
  QDA_Chol1                : OK        (accuracy=0.8805)
  QDA_Chol2                : OK        (accuracy=0.8805)
  QDA_Chol3                : OK        (accuracy=0.8805)
  TensorizedChol           : OK        (accuracy=0.8805)
  EfficientChol            : OK        (accuracy=0.8805)
